## Célula 1 — Instalação de dependências

Garante que todas as bibliotecas necessárias estão instaladas no ambiente.

In [ ]:
# Execute esta célula apenas uma vez para instalar as dependências
# O '!' permite rodar comandos de terminal direto no Jupyter
#%pip install feedparser google-api-python-client isodate openpyxl python-dotenv --quiet

## Célula 2 — Imports

Carregamos todas as bibliotecas que serão usadas ao longo do notebook.

In [ ]:
from __future__ import annotations

import os
import logging
from pathlib import Path
from datetime import datetime, timezone
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import Dict, Iterable, List

import feedparser
import isodate
import pandas as pd
from dotenv import load_dotenv
from googleapiclient.discovery import build

# ---------------------------------------------------------------------------
# Configuração de log
# Usamos logging ao invés de print() para ter controle de nível (INFO, WARNING, ERROR)
# Isso é especialmente útil em execuções longas: sabemos exatamente o que aconteceu
# ---------------------------------------------------------------------------
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)

print("✅ Imports OK")

✅ Imports OK


## Célula 3 — Configuração via `.env`

### Por que `.env` e não hardcode?

A API Key **nunca** deve ficar no código-fonte.  
Se você versionar o projeto com Git, a chave ficaria exposta publicamente.  

Crie um arquivo `.env` na mesma pasta do notebook com o conteúdo:
```
YOUTUBE_API_KEY=sua_chave_aqui
```
O `python-dotenv` lê esse arquivo e injeta as variáveis no ambiente.

In [ ]:
# Carrega as variáveis do arquivo .env para o ambiente
load_dotenv()

API_KEY: str = os.environ.get("YOUTUBE_API_KEY", "")

# Validação antecipada — melhor falhar cedo do que no meio da execução
if not API_KEY:
    raise EnvironmentError(
        "❌ Variável YOUTUBE_API_KEY não encontrada. "
        "Crie um arquivo .env com: YOUTUBE_API_KEY=sua_chave"
    )

print(f"✅ API Key carregada: {'*' * (len(API_KEY) - 4)}{API_KEY[-4:]}")

✅ API Key carregada: ***********************************6jF4


## Célula 4 — Parâmetros de execução

Centralizamos aqui tudo que o usuário pode querer ajustar.  
Isso evita que parâmetros fiquem espalhados pelo código.

In [ ]:
# ──────────────────────────────────────────────
# Parâmetros — edite conforme sua necessidade
# ──────────────────────────────────────────────

CANAIS: List[str] = [
    "UC3Z2XdKUu21_KtMsohZedOQ", "UCN0W0kG9zolEDAn228CvFSQ", "UCCWSZHyZmONoOwb76uMkccg", "UCIRQQEextXQEV18j9kedBOw", "UC-ezuc6BSkMhdPIoaEU8S1g",
    "UCrNt1hZdMx-nFk-tqXsk-GQ", "UCnBNoed6NaPlpL8blj1yy8Q", "UCZE5espnlDl2dDRXD2XLMLg", "UCIt_jZw64ryRDpFW1w-m4vA", "UCcLvYJo2ke6Ckx4CqsEXyRQ",
    "UCkPwI3gaSr65levrx1j9okQ", "UCwLxXLLWEIJFHEeTMlYqHTA", "UCrp0zmecZ3TNV8FSR-tjv7A", "UCD_q9m3EPyFZZ0d3XWcrqWQ", "UCIeL1JF5Q7ALE1qOGPJBm5w",
    "UCj3jEZuLmXOndQJFkPTp3Nw", "UCBX4Z7RAs8ktgsyXm4ES_dg", "UC5Fl2W965fyBCv4gLb-U2hg", "UC_g73GNqsyBWvEehY0lNZBA", "UCD89tNSy9kC6QBjFAcgVd3g",
    "UC-hA65Fjv5X8h-J92zc_L8Q", "UCzLAzI6Q-0WX2IbKfLmtZUw", "UCfMKycqfaMdj9-jVDUuU15Q", "UCn_3RiYrABm3o7puJjTZO3w", "UCGnqRqH9I2v-rNdmXmHmHeg",
    "UChBtPExX9RjCdmpAizK7ccQ", "UC6pkw1C4ha1cmU9aDJ8OxUw", "UCRekFQ2mNTWbMwvH7JdD3xA", "UCQlVe5fDZN99GuRnBKzwn4w", "UCOn6-uNuXeXJqmbywtfBgdQ",
    "UCS4ZSVzsXgDiXV7Pwro4a0A", "UClmrZELCfAbZNzpl6rAT0gw", "UCHGl5Nn0GfMETBgjIkJDWjw", "UCD_q9m3EPyFZZ0d3XWcrqWQ", "UCteJrv6W7MWB5SoRvfjF36Q",
    "UCnTTl5-22qOKRSVmuGwm1Lw", "UCHniugO-y89ZBnaJUCn57Iw", "UCSXdP8V4jRaw-8YMTpa6glw", "UCN0W0kG9zolEDAn228CvFSQ", "UCiuihFCo8gfLInRK55F914g",
    "UCPUb-aMI0HiBjC1rsuRN4ng", "UC3XQ0w18WsYV3vf5E6E7D0w", "UCKjV2NaruwIaBORr9MCcAdw", "UCLLV2I1PKPgDQYocS1nVMpg", "UCWTXUWC88wp6Xzr5fTpGEBQ",
    "UCT3iQIbLMQc_wsj87KwlZWA", "UC3XQ0w18WsYV3vf5E6E7D0w", "UCKjV2NaruwIaBORr9MCcAdw", "UCLLV2I1PKPgDQYocS1nVMpg", "UCWTXUWC88wp6Xzr5fTpGEBQ",
]

MAX_POR_CANAL: int = 3        # quantos vídeos buscar por canal via RSS
LIMITE_SHORTS_SEG: int = 60    # vídeos com duração <= 60s são considerados Shorts
MAX_WORKERS: int = 8           # threads simultâneas para buscar RSS
SAIDA_XLSX: Path = Path("saida/videos_apenas.xlsx")  # caminho de saída

print(f"✅ {len(CANAIS)} canal(is) configurado(s)")
print(f"   Máx. vídeos por canal : {MAX_POR_CANAL}")
print(f"   Limite Shorts         : <= {LIMITE_SHORTS_SEG}s")
print(f"   Workers (RSS paralelo): {MAX_WORKERS}")
print(f"   Saída                 : {SAIDA_XLSX}")

✅ 50 canal(is) configurado(s)
   Máx. vídeos por canal : 3
   Limite Shorts         : <= 60s
   Workers (RSS paralelo): 8
   Saída                 : saida\videos_apenas.xlsx


## Célula 5 — Cliente da YouTube Data API

### Melhoria: instanciar o cliente UMA única vez

Na versão original, `build()` era chamado dentro da função `duracoes_por_video_id()`.  
Isso significa que toda vez que a função fosse chamada, um novo cliente HTTP seria criado.

Aqui criamos o cliente **uma vez** e passamos ele como argumento — mais eficiente e testável.

In [ ]:
# Criamos o cliente da API aqui, fora das funções
# Ele será reutilizado em todas as chamadas subsequentes
yt_client = build("youtube", "v3", developerKey=API_KEY)

print("✅ Cliente da YouTube Data API criado com sucesso")

19:08:44  INFO      file_cache is only supported with oauth2client<4.0.0


✅ Cliente da YouTube Data API criado com sucesso


## Célula 6 — Descobrir `channel_id` a partir do `@handle`

URLs do YouTube modernas usam o formato `youtube.com/@NomeDoCanal`.  
O `@handle` **não é** o `channel_id` — precisamos converter um no outro.

### Como funciona
Usamos o endpoint `channels().list(forHandle=...)` da YouTube Data API.  
Ele aceita o handle **com ou sem** o `@` e retorna o `channel_id` correspondente.

### Custo de cota
Cada chamada consome **1 unidade** de cota da API.  
Para 30 canais = 30 unidades (limite diário padrão: 10.000 unidades). Sem problema.

> ⚠️ **Execute esta célula apenas quando precisar descobrir novos IDs.**  
> Depois de ter os `channel_id`, cole-os na Célula 4 (`CANAIS`) e não precisa rodar esta de novo.

In [ ]:
# ── Lista de handles para converter ───────────────────────────────────────
# Cole aqui os @handles dos canais — com ou sem o '@'
HANDLES_PARA_CONVERTER: List[str] = [
    "ACaraDaRiquezaOficial",
    "ramosdedinheiro",
    "academia.fundos.imobili%C3%A1rios",
    "alexfiis",
    "rendacomdividendoss",
    "ClickInvest",
    "ClubedoValor",
    "danimengue",
    "diaadiainvestindo",
    "dicionariodoinvestidor",
    "DinheiroComVoce",
    "economirna",
    "economistasincero",
    "matheusefeitoinvestimentos",
    "CanalElaInveste",
    "seudividendo",
    "neto.pereira",
    "financeologia",
    "FonteDaFortuna",
    "GemeosInvestem",
    "geracaodividendoss",
    "HannahGuarilha",
    "oinvestidorclassemedia",
    "Investirecrescer2024",
    "IrmaosDiasPodcast",
    "JovensDeNegocios",
    "JovensInvestem",
    "KellyInveste",
    "LiberdadeELogoAli",
    "LiberdadeFinanceiracomFiis",
    "lucasfiis",
    "LucroFC",
    "matheusefeitoinvestimentos",
    "MestreDaRiqueza",
    "onebertofii",
    "pequenoinvestidoroficial",
    "ProfessorBaroni",
    "ramosdedinheiro",
    "RiquezaEmDias",
    "robcorrearesearch",
    "SeBanque",
    "OSilvioPaulo",
    "stellamourao",
    "thomazxavier",
    "TioFiis",
    "TubaraoDaBolsa",
    "brunoperini",
]

# ── Busca e exibe os channel_ids ───────────────────────────────────────────
print(f"Buscando channel_id para {len(HANDLES_PARA_CONVERTER)} handle(s)...\n")

resultados_handles = []  # guardamos os resultados para inspecionar depois

for handle in HANDLES_PARA_CONVERTER:
    # Garantimos que o handle não tem '@' duplicado antes de enviar
    handle_limpo = handle.lstrip("@")

    try:
        resp = (
            yt_client.channels()
            .list(part="id,snippet", forHandle=handle_limpo)
            .execute()
        )
        items = resp.get("items", [])

        if items:
            channel_id   = items[0]["id"]
            channel_name = items[0]["snippet"]["title"]
            print(f"✅  @{handle_limpo:<35} → {channel_id}  ({channel_name})")
            resultados_handles.append({
                "handle": handle_limpo,
                "channel_id": channel_id,
                "channel_title": channel_name,
                "erro": None,
            })
        else:
            print(f"❌  @{handle_limpo:<35} → não encontrado")
            resultados_handles.append({
                "handle": handle_limpo,
                "channel_id": None,
                "channel_title": None,
                "erro": "handle não encontrado",
            })

    except Exception as exc:  # noqa: BLE001
        print(f"❌  @{handle_limpo:<35} → erro na API: {exc}")
        resultados_handles.append({
            "handle": handle_limpo,
            "channel_id": None,
            "channel_title": None,
            "erro": str(exc),
        })

# ── Exibe tabela com todos os resultados ──────────────────────────────────
print("\n── Tabela completa ──")
df_handles = pd.DataFrame(resultados_handles)
df_handles

Buscando channel_id para 47 handle(s)...

✅  @ACaraDaRiquezaOficial               → UC3Z2XdKUu21_KtMsohZedOQ  (A Cara da Riqueza)
✅  @ramosdedinheiro                     → UCN0W0kG9zolEDAn228CvFSQ  (Ramos de Dinheiro)
✅  @academia.fundos.imobili%C3%A1rios   → UCCWSZHyZmONoOwb76uMkccg  (Academia de Fundos Imobiliários )
✅  @alexfiis                            → UCIRQQEextXQEV18j9kedBOw  (Alex Fiis)
✅  @rendacomdividendoss                 → UC-ezuc6BSkMhdPIoaEU8S1g  (Renda Com Dividendos | Lucas Santos)
✅  @ClickInvest                         → UCrNt1hZdMx-nFk-tqXsk-GQ  (ClickInvest)
✅  @ClubedoValor                        → UCnBNoed6NaPlpL8blj1yy8Q  (Clube do Valor)
✅  @danimengue                          → UCZE5espnlDl2dDRXD2XLMLg  (Dani Mengue)
✅  @diaadiainvestindo                   → UCIt_jZw64ryRDpFW1w-m4vA  (Dia a Dia Investindo)
✅  @dicionariodoinvestidor              → UCcLvYJo2ke6Ckx4CqsEXyRQ  (Dicionário do Investidor)
✅  @DinheiroComVoce                     → UCkPwI3gaSr65lev

,handle,channel_id,channel_title,erro
0,ACaraDaRiquezaOficial,UC3Z2XdKUu21_KtMsohZedOQ,A Cara da Riqueza,None
1,ramosdedinheiro,UCN0W0kG9zolEDAn228CvFSQ,Ramos de Dinheiro,None
2,academia.fundos.imobili%C3%A1rios,UCCWSZHyZmONoOwb76uMkccg,Academia de Fundos Imobiliários,None
3,alexfiis,UCIRQQEextXQEV18j9kedBOw,Alex Fiis,None
4,rendacomdividendoss,UC-ezuc6BSkMhdPIoaEU8S1g,Renda Com Dividendos | Lucas Santos,None
5,ClickInvest,UCrNt1hZdMx-nFk-tqXsk-GQ,ClickInvest,None
6,ClubedoValor,UCnBNoed6NaPlpL8blj1yy8Q,Clube do Valor,None
7,danimengue,UCZE5espnlDl2dDRXD2XLMLg,Dani Mengue,None
8,diaadiainvestindo,UCIt_jZw64ryRDpFW1w-m4vA,Dia a Dia Investindo,None
9,dicionariodoinvestidor,UCcLvYJo2ke6Ckx4CqsEXyRQ,Dicionário do Investidor,None


## Célula 7 — Função: buscar um canal via RSS

Separamos a lógica de **um único canal** em uma função própria.  
Isso facilita a paralelização: podemos chamar essa função para N canais ao mesmo tempo.

In [ ]:
def _buscar_canal_rss(channel_id: str, max_por_canal: int) -> List[Dict]:
    """
    Busca vídeos de UM único canal via RSS.

    Separamos em função própria para que o ThreadPoolExecutor
    possa chamá-la em paralelo para múltiplos canais.

    Retorna uma lista de dicionários com metadados básicos.
    Em caso de erro no feed, retorna um item com o campo 'error' preenchido.
    """
    url = f"https://www.youtube.com/feeds/videos.xml?channel_id={channel_id}"
    feed = feedparser.parse(url)

    # feedparser sinaliza erros de parsing no atributo 'bozo'
    if getattr(feed, "bozo", 0):
        erro = str(getattr(feed, "bozo_exception", "Erro desconhecido ao ler RSS"))
        log.warning("Erro no feed do canal %s: %s", channel_id, erro)
        return [{
            "channel_id": channel_id,
            "channel_title": None,
            "video_id": None,
            "video_title": None,
            "video_url": None,
            "published": None,
            "error": erro,
        }]

    channel_title = getattr(feed.feed, "title", None)
    itens: List[Dict] = []

    for entry in feed.entries[:max_por_canal]:
        # O RSS do YouTube expõe o video_id no campo 'yt_videoid'
        # Removemos a linha redundante 'video_id = None' que existia no original
        video_id = entry.get("yt_videoid") or entry.get("yt:videoid")

        # Conversão de data: preferimos published_parsed (já parseado pelo feedparser)
        # e só usamos o campo 'published' (string) como fallback
        if getattr(entry, "published_parsed", None):
            dt = datetime(*entry.published_parsed[:6], tzinfo=timezone.utc)
            published_iso = dt.isoformat()
        else:
            published_iso = entry.get("published")

        itens.append({
            "channel_id": channel_id,
            "channel_title": channel_title,
            "video_id": video_id,
            "video_title": entry.get("title"),
            "video_url": entry.get("link"),
            "published": published_iso,
            "error": None,
        })

    log.info("Canal '%s' → %d vídeo(s) encontrado(s)", channel_title or channel_id, len(itens))
    return itens


print("✅ Função _buscar_canal_rss definida")

✅ Função _buscar_canal_rss definida


## Célula 8 — Função: buscar todos os canais em paralelo

### Melhoria: `ThreadPoolExecutor` para paralelização

Na versão original, os canais eram processados **um por um** (loop `for` sequencial).  
Cada `feedparser.parse()` é uma requisição HTTP bloqueante.

Com `ThreadPoolExecutor`, disparamos até `MAX_WORKERS` requisições **ao mesmo tempo**.  
Para 20 canais com latência de ~1s cada: **20s → ~3s** (dependendo da rede).

In [ ]:
def buscar_ids_rss(
    channel_ids: Iterable[str],
    max_por_canal: int = 10,
    max_workers: int = 8,
) -> List[Dict]:
    """
    Busca vídeos de TODOS os canais em paralelo usando threads.

    Por que threads e não multiprocessing?
    - A operação é I/O bound (espera de rede), não CPU bound
    - Threads são suficientes e mais leves para esse caso
    - O GIL do Python não é problema aqui

    Retorna lista com todos os vídeos de todos os canais.
    """
    ids = list(channel_ids)
    todos: List[Dict] = []

    log.info("Iniciando busca RSS para %d canal(is) com %d workers...", len(ids), max_workers)

    # submit() envia cada tarefa para o pool de threads
    # as_completed() itera sobre as futures à medida que terminam
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {
            executor.submit(_buscar_canal_rss, cid, max_por_canal): cid
            for cid in ids
        }

        for future in as_completed(futures):
            channel_id = futures[future]
            try:
                resultado = future.result()
                todos.extend(resultado)
            except Exception as exc:  # noqa: BLE001
                # Captura erros inesperados (timeout, etc.) sem derrubar tudo
                log.error("Falha inesperada no canal %s: %s", channel_id, exc)
                todos.append({
                    "channel_id": channel_id,
                    "channel_title": None,
                    "video_id": None,
                    "video_title": None,
                    "video_url": None,
                    "published": None,
                    "error": str(exc),
                })

    log.info("Total de itens coletados: %d", len(todos))
    return todos


print("✅ Função buscar_ids_rss definida")

✅ Função buscar_ids_rss definida


## Célula 9 — Função: buscar durações via YouTube Data API

### Melhorias aplicadas:
1. **Recebe o cliente** como parâmetro (não cria um novo a cada chamada)
2. **`try/except` por item**: um vídeo com dado faltando não interrompe o lote inteiro
3. **Log de progresso** para lotes grandes

In [ ]:
def duracoes_por_video_id(
    cliente,
    video_ids: List[str],
) -> Dict[str, int]:
    """
    Retorna {video_id: duração_em_segundos} para todos os IDs informados.

    A YouTube Data API aceita no máximo 50 IDs por requisição,
    então processamos em lotes de 50.

    Parâmetros
    ----------
    cliente   : objeto retornado por googleapiclient.discovery.build()
    video_ids : lista de IDs de vídeos do YouTube
    """
    out: Dict[str, int] = {}
    total_lotes = (len(video_ids) + 49) // 50  # arredonda para cima

    for i in range(0, len(video_ids), 50):
        lote = video_ids[i : i + 50]
        num_lote = i // 50 + 1
        log.info("Buscando durações — lote %d/%d (%d vídeos)", num_lote, total_lotes, len(lote))

        try:
            resp = (
                cliente.videos()
                .list(part="contentDetails", id=",".join(lote))
                .execute()
            )
        except Exception as exc:  # noqa: BLE001
            # Erro na chamada HTTP do lote inteiro → loga e pula
            log.error("Erro na API para o lote %d: %s", num_lote, exc)
            continue

        for item in resp.get("items", []):
            vid = item["id"]
            try:
                # A duração vem no formato ISO 8601, ex: 'PT4M13S'
                # isodate.parse_duration converte para timedelta
                iso_dur = item["contentDetails"]["duration"]
                seconds = int(isodate.parse_duration(iso_dur).total_seconds())
                out[vid] = seconds
            except (KeyError, AttributeError, ValueError) as exc:
                # Vídeo com dado malformado → registra aviso e continua
                log.warning("Não foi possível parsear duração de '%s': %s", vid, exc)

    log.info("Durações obtidas: %d/%d vídeos", len(out), len(video_ids))
    return out


print("✅ Função duracoes_por_video_id definida")

✅ Função duracoes_por_video_id definida


## Célula 10 — Função: filtrar Shorts

### Melhoria: lógica do NaN corrigida

Na versão original:
```python
df[(df["duration_seconds"].isna()) | (df["duration_seconds"] > limite)]
```
Isso **mantinha** vídeos sem duração (NaN) — potenciais Shorts não filtrados.

Na versão corrigida, descartamos qualquer vídeo sem duração confirmada.

In [ ]:
def somente_videos_nao_shorts(
    df: pd.DataFrame,
    cliente,
    limite_shorts_seg: int = 60,
) -> pd.DataFrame:
    """
    Enriquece o DataFrame com a duração de cada vídeo
    e remove Shorts (duração <= limite_shorts_seg).

    Vídeos sem duração retornada pela API são DESCARTADOS
    (comportamento seguro: melhor perder um vídeo do que incluir um Short).
    """
    df = df.copy()

    ids = df["video_id"].dropna().astype(str).unique().tolist()
    if not ids:
        log.warning("Nenhum video_id válido encontrado — retornando DataFrame vazio.")
        return df

    log.info("%d video_id(s) únicos para consultar na API", len(ids))

    dur = duracoes_por_video_id(cliente, ids)
    df["duration_seconds"] = df["video_id"].map(dur)

    antes = len(df)

    # Corrigido: exigimos que duration_seconds seja > limite
    # NaN (sem duração confirmada) é DESCARTADO
    df = df[df["duration_seconds"] > limite_shorts_seg]

    removidos = antes - len(df)
    log.info("%d Shorts/sem-duração removidos. Restam %d vídeo(s).", removidos, len(df))

    return df


print("✅ Função somente_videos_nao_shorts definida")

✅ Função somente_videos_nao_shorts definida


## Célula 11 — Função: salvar Excel

Mantida praticamente igual à original — já estava bem implementada.  
Adicionamos apenas log de confirmação.

In [ ]:
def salvar_excel(
    df: pd.DataFrame,
    path_xlsx: str | Path,
    sheet_name: str = "videos",
) -> None:
    """
    Salva o DataFrame em arquivo .xlsx.

    O Excel não aceita datas com timezone, então convertemos
    para UTC e depois removemos a informação de fuso.
    """
    path_xlsx = Path(path_xlsx)
    path_xlsx.parent.mkdir(parents=True, exist_ok=True)

    if "published" in df.columns:
        df = df.copy()
        df["published"] = (
            pd.to_datetime(df["published"], errors="coerce", utc=True)
            .dt.tz_convert(None)  # remove timezone para compatibilidade com Excel
        )

    with pd.ExcelWriter(path_xlsx, engine="openpyxl") as writer:
        df.to_excel(writer, index=False, sheet_name=sheet_name)

    log.info("✅ Arquivo salvo: %s  (%d linhas)", path_xlsx, len(df))


print("✅ Função salvar_excel definida")

✅ Função salvar_excel definida


## Célula 12 — Execução principal

Aqui orquestramos tudo na sequência correta, com logs intermediários para acompanhamento.

In [ ]:
# ── Passo 1: buscar vídeos via RSS (em paralelo) ───────────────────────────
itens = buscar_ids_rss(
    channel_ids=CANAIS,
    max_por_canal=MAX_POR_CANAL,
    max_workers=MAX_WORKERS,
)

df = pd.DataFrame(itens)
print(f"\nDataFrame inicial: {len(df)} linha(s)")
df.head()

19:08:49  INFO      Iniciando busca RSS para 50 canal(is) com 8 workers...
19:08:49  INFO      Canal 'A Cara da Riqueza' → 3 vídeo(s) encontrado(s)
19:08:49  INFO      Canal 'Clube do Valor' → 3 vídeo(s) encontrado(s)
19:08:49  INFO      Canal 'Academia de Fundos Imobiliários' → 3 vídeo(s) encontrado(s)
19:08:49  INFO      Canal 'Ramos de Dinheiro' → 3 vídeo(s) encontrado(s)
19:08:49  INFO      Canal 'Dinheiro Com Você - Por William Ribeiro' → 3 vídeo(s) encontrado(s)
19:08:49  INFO      Canal 'EconoMirna' → 3 vídeo(s) encontrado(s)
19:08:50  INFO      Canal 'Economista Sincero' → 3 vídeo(s) encontrado(s)
19:08:50  INFO      Canal 'ClickInvest' → 3 vídeo(s) encontrado(s)
19:08:50  INFO      Canal 'Dani Mengue' → 3 vídeo(s) encontrado(s)
19:08:50  INFO      Canal 'Alex Fiis' → 3 vídeo(s) encontrado(s)
19:08:50  INFO      Canal 'Renda Com Dividendos | Lucas Santos' → 3 vídeo(s) encontrado(s)
19:08:50  INFO      Canal 'Ela Investe' → 3 vídeo(s) encontrado(s)
19:08:50  INFO      Canal 'Dia


DataFrame inicial: 147 linha(s)


,channel_id,channel_title,video_id,video_title,video_url,published,error
0,UC3Z2XdKUu21_KtMsohZedOQ,A Cara da Riqueza,HAoPvnlmk54,O que você consegue refletir vendo um vídeo de...,https://www.youtube.com/shorts/HAoPvnlmk54,2026-03-16T21:04:17+00:00,None
1,UC3Z2XdKUu21_KtMsohZedOQ,A Cara da Riqueza,C2rJr0-ER3A,GARE11 CAINDO MUITO: o problema é SÓ o INQUILI...,https://www.youtube.com/watch?v=C2rJr0-ER3A,2026-03-13T21:12:18+00:00,None
2,UC3Z2XdKUu21_KtMsohZedOQ,A Cara da Riqueza,mokwPrXEgrY,GARE11 registra queda de 10%! O que isso signi...,https://www.youtube.com/shorts/mokwPrXEgrY,2026-03-13T20:47:06+00:00,None
3,UCnBNoed6NaPlpL8blj1yy8Q,Clube do Valor,o-azylpJLqI,TODO MUNDO DESISTIU… MAS ESTOU COMPRANDO ESSAS...,https://www.youtube.com/watch?v=o-azylpJLqI,2026-03-16T21:00:52+00:00,None
4,UCnBNoed6NaPlpL8blj1yy8Q,Clube do Valor,yodCc3vE4oA,O Rei dos Bares Cariocas... #educaçãofinancei...,https://www.youtube.com/shorts/yodCc3vE4oA,2026-03-16T20:00:56+00:00,None


In [ ]:
# ── Passo 2: filtrar Shorts via duração (YouTube Data API) ─────────────────
df = somente_videos_nao_shorts(
    df,
    cliente=yt_client,
    limite_shorts_seg=LIMITE_SHORTS_SEG,
)

print(f"\nDataFrame após filtro de Shorts: {len(df)} linha(s)")
df.head()

19:08:52  INFO      129 video_id(s) únicos para consultar na API
19:08:52  INFO      Buscando durações — lote 1/3 (50 vídeos)
19:08:52  WARNING   Não foi possível parsear duração de 'p00Xktc1rao': 'duration'
19:08:52  INFO      Buscando durações — lote 2/3 (50 vídeos)
19:08:53  WARNING   Não foi possível parsear duração de 'iqmiSDsfGdk': 'duration'
19:08:53  INFO      Buscando durações — lote 3/3 (29 vídeos)
19:08:53  INFO      Durações obtidas: 127/129 vídeos
19:08:53  INFO      19 Shorts/sem-duração removidos. Restam 128 vídeo(s).



DataFrame após filtro de Shorts: 128 linha(s)


,channel_id,channel_title,video_id,video_title,video_url,published,error,duration_seconds
0,UC3Z2XdKUu21_KtMsohZedOQ,A Cara da Riqueza,HAoPvnlmk54,O que você consegue refletir vendo um vídeo de...,https://www.youtube.com/shorts/HAoPvnlmk54,2026-03-16T21:04:17+00:00,None,85.0
1,UC3Z2XdKUu21_KtMsohZedOQ,A Cara da Riqueza,C2rJr0-ER3A,GARE11 CAINDO MUITO: o problema é SÓ o INQUILI...,https://www.youtube.com/watch?v=C2rJr0-ER3A,2026-03-13T21:12:18+00:00,None,1242.0
2,UC3Z2XdKUu21_KtMsohZedOQ,A Cara da Riqueza,mokwPrXEgrY,GARE11 registra queda de 10%! O que isso signi...,https://www.youtube.com/shorts/mokwPrXEgrY,2026-03-13T20:47:06+00:00,None,153.0
3,UCnBNoed6NaPlpL8blj1yy8Q,Clube do Valor,o-azylpJLqI,TODO MUNDO DESISTIU… MAS ESTOU COMPRANDO ESSAS...,https://www.youtube.com/watch?v=o-azylpJLqI,2026-03-16T21:00:52+00:00,None,840.0
4,UCnBNoed6NaPlpL8blj1yy8Q,Clube do Valor,yodCc3vE4oA,O Rei dos Bares Cariocas... #educaçãofinancei...,https://www.youtube.com/shorts/yodCc3vE4oA,2026-03-16T20:00:56+00:00,None,144.0


In [ ]:
# ── Passo 3: organizar colunas e exportar ──────────────────────────────────
COLUNAS_FINAIS = [
    "channel_title",
    "published",
    "duration_seconds",
    "video_title",
    "video_url",
    "video_id",
    "channel_id",
    "error",
]

# Filtramos apenas as colunas que existem no DataFrame
# (evita KeyError se alguma coluna não foi gerada)
df = df[[c for c in COLUNAS_FINAIS if c in df.columns]]

salvar_excel(df, SAIDA_XLSX)

print("\n✅ Pipeline concluído com sucesso!")
df

19:08:53  INFO      ✅ Arquivo salvo: saida\videos_apenas.xlsx  (128 linhas)



✅ Pipeline concluído com sucesso!


,channel_title,published,duration_seconds,video_title,video_url,video_id,channel_id,error
0,A Cara da Riqueza,2026-03-16T21:04:17+00:00,85.0,O que você consegue refletir vendo um vídeo de...,https://www.youtube.com/shorts/HAoPvnlmk54,HAoPvnlmk54,UC3Z2XdKUu21_KtMsohZedOQ,None
1,A Cara da Riqueza,2026-03-13T21:12:18+00:00,1242.0,GARE11 CAINDO MUITO: o problema é SÓ o INQUILI...,https://www.youtube.com/watch?v=C2rJr0-ER3A,C2rJr0-ER3A,UC3Z2XdKUu21_KtMsohZedOQ,None
2,A Cara da Riqueza,2026-03-13T20:47:06+00:00,153.0,GARE11 registra queda de 10%! O que isso signi...,https://www.youtube.com/shorts/mokwPrXEgrY,mokwPrXEgrY,UC3Z2XdKUu21_KtMsohZedOQ,None
3,Clube do Valor,2026-03-16T21:00:52+00:00,840.0,TODO MUNDO DESISTIU… MAS ESTOU COMPRANDO ESSAS...,https://www.youtube.com/watch?v=o-azylpJLqI,o-azylpJLqI,UCnBNoed6NaPlpL8blj1yy8Q,None
4,Clube do Valor,2026-03-16T20:00:56+00:00,144.0,O Rei dos Bares Cariocas... #educaçãofinancei...,https://www.youtube.com/shorts/yodCc3vE4oA,yodCc3vE4oA,UCnBNoed6NaPlpL8blj1yy8Q,None
...,...,...,...,...,...,...,...,...
141,Academia Thomaz Xavier de Finanças,2026-03-15T15:00:03+00:00,90.0,SNAG11 VALE A PENA? FIAGRO muito bem DIVERSIFI...,https://www.youtube.com/shorts/aiWokP4VIY4,aiWokP4VIY4,UCT3iQIbLMQc_wsj87KwlZWA,None
142,Academia Thomaz Xavier de Finanças,2026-03-14T20:00:13+00:00,1234.0,SNEL11 MUDOU A SUA ESTRATÉGIA? Vale a pena inv...,https://www.youtube.com/watch?v=OMZ9dQoYFfk,OMZ9dQoYFfk,UCT3iQIbLMQc_wsj87KwlZWA,None
144,Stella Mourão FIIs,2026-03-16T15:01:17+00:00,577.0,#MXRF11 - COTA CAI MAS SEGUE CARA!,https://www.youtube.com/watch?v=gfDyj6uNcbc,gfDyj6uNcbc,UCWTXUWC88wp6Xzr5fTpGEBQ,None
145,Stella Mourão FIIs,2026-03-15T15:00:40+00:00,434.0,#HSML11 - FEZ RESERVA!,https://www.youtube.com/watch?v=QVBAifKjt4Y,QVBAifKjt4Y,UCWTXUWC88wp6Xzr5fTpGEBQ,None
